# Walk-forward results

Per-fold predictive metrics from embargoed walk-forward validation.
Load `data/results/xgb_wf.parquet` and `data/results/lstm_wf.parquet` from `scripts/run_walkforward.py`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)

if Path.cwd().name == "notebooks":
    os.chdir("..")


def _load_preds(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"missing {path}")
        return None
    df = pd.read_parquet(path)
    print(f"{path.name}: rows={len(df):,} folds={sorted(df['fold'].unique())}")
    return df


preds_xgb = _load_preds(Path("data/results/xgb_wf.parquet"))
preds_lstm = _load_preds(Path("data/results/lstm_wf.parquet"))
if preds_xgb is None:
    raise FileNotFoundError("Need xgb_wf.parquet — run: .venv/bin/python scripts/run_walkforward.py --model xgb")

In [ ]:
def _signed_to_cls(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y).astype(np.int64)
    out = np.empty_like(y)
    out[y == -1] = 0
    out[y == 0] = 1
    out[y == 1] = 2
    return out


def fold_metrics(g: pd.DataFrame) -> dict:
    y_cls = _signed_to_cls(g["y_true"].to_numpy())
    probs = np.column_stack(
        [g["prob_down"].to_numpy(), g["prob_zero"].to_numpy(), g["prob_up"].to_numpy()]
    )
    y_pred = probs.argmax(axis=1)
    mask = y_cls != 1
    if mask.sum() >= 2 and len(np.unique(y_cls[mask])) == 2:
        auc = float(roc_auc_score((y_cls[mask] == 2).astype(int), probs[mask, 2]))
    else:
        auc = float("nan")
    return {
        "fold": int(g["fold"].iloc[0]),
        "accuracy": float(accuracy_score(y_cls, y_pred)),
        "log_loss": float(log_loss(y_cls, probs, labels=[0, 1, 2])),
        "macro_f1": float(
            f1_score(y_cls, y_pred, average="macro", labels=[0, 1, 2], zero_division=0)
        ),
        "auc_up_vs_down": auc,
        "n": len(g),
        "test_start": pd.Timestamp(g["ts_event"].min()),
    }


def metrics_table(preds: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([fold_metrics(g) for _, g in preds.groupby("fold", sort=True)])


metrics_xgb = metrics_table(preds_xgb)
metrics_lstm = metrics_table(preds_lstm) if preds_lstm is not None else None
metrics_xgb

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].plot(metrics_xgb["test_start"], metrics_xgb["accuracy"], marker="o", label="XGBoost")
axes[1].plot(metrics_xgb["test_start"], metrics_xgb["auc_up_vs_down"], marker="o", label="XGBoost")

if metrics_lstm is not None:
    axes[0].plot(metrics_lstm["test_start"], metrics_lstm["accuracy"], marker="s", label="LSTM")
    axes[1].plot(
        metrics_lstm["test_start"], metrics_lstm["auc_up_vs_down"], marker="s", label="LSTM"
    )

axes[0].axhline(metrics_xgb["accuracy"].mean(), color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("Per-fold accuracy")
axes[0].set_ylabel("accuracy")
axes[0].legend()

axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="chance")
axes[1].set_title("Per-fold AUC (up vs down)")
axes[1].set_ylabel("AUC")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("test-day start")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Walk-forward — XGBoost vs LSTM", y=1.02)
fig.tight_layout()

fig_dir = Path("notebooks/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "wf_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"XGB  acc={metrics_xgb['accuracy'].mean():.4f}±{metrics_xgb['accuracy'].std():.4f}  "
    f"auc={metrics_xgb['auc_up_vs_down'].mean():.4f}±{metrics_xgb['auc_up_vs_down'].std():.4f}"
)
if metrics_lstm is not None:
    print(
        f"LSTM acc={metrics_lstm['accuracy'].mean():.4f}±{metrics_lstm['accuracy'].std():.4f}  "
        f"auc={metrics_lstm['auc_up_vs_down'].mean():.4f}±{metrics_lstm['auc_up_vs_down'].std():.4f}"
    )

In [ ]:
preds_random = _load_preds(Path("data/results/xgb_wf_random_labels.parquet"))
if preds_lstm is None:
    raise FileNotFoundError("Need lstm_wf.parquet — run: .venv/bin/python scripts/run_walkforward.py --model lstm")
if preds_random is None:
    raise FileNotFoundError("Need xgb_wf_random_labels.parquet — run: .venv/bin/python scripts/leakage_sanity.py")

In [ ]:
def always_zero_fold_metrics(g: pd.DataFrame) -> dict:
  """Naive baseline: always predict label 0 (neutral)."""
  y_cls = _signed_to_cls(g["y_true"].to_numpy())
  y_pred = np.ones_like(y_cls)
  probs = np.zeros((len(y_cls), 3), dtype=np.float64)
  probs[:, 1] = 1.0
  mask = y_cls != 1
  if mask.sum() >= 2 and len(np.unique(y_cls[mask])) == 2:
    auc = float(roc_auc_score((y_cls[mask] == 2).astype(int), probs[mask, 2]))
  else:
    auc = float("nan")
  return {
    "accuracy": float(accuracy_score(y_cls, y_pred)),
    "log_loss": float(log_loss(y_cls, probs, labels=[0, 1, 2])),
    "macro_f1": float(f1_score(y_cls, y_pred, average="macro", labels=[0, 1, 2], zero_division=0)),
    "auc_up_vs_down": auc,
  }


def aggregate_row(per_fold: pd.DataFrame) -> dict:
  return {
    "mean_accuracy": float(per_fold["accuracy"].mean()),
    "mean_macro_f1": float(per_fold["macro_f1"].mean()),
    "mean_log_loss": float(per_fold["log_loss"].mean()),
    "mean_auc": float(per_fold["auc_up_vs_down"].mean()),
    "std_accuracy": float(per_fold["accuracy"].std()),
  }


summary_rows = []
for name, preds, fn in [
  ("XGBoost", preds_xgb, fold_metrics),
  ("LSTM", preds_lstm, fold_metrics),
  ("Always-zero baseline", preds_xgb, always_zero_fold_metrics),
  ("Random-label control (XGB)", preds_random, fold_metrics),
]:
  per_fold = pd.DataFrame([fn(g) for _, g in preds.groupby("fold", sort=True)])
  summary_rows.append({"model": name, **aggregate_row(per_fold)})

summary = pd.DataFrame(summary_rows)
summary

In [ ]:
summary_path = Path("data/results/summary.csv")
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_path, index=False)
print(f"Saved {summary_path}")

# Enhanced figure: per-fold lines + aggregate bars (same as scripts/summarize_walkforward.py)
from scripts.summarize_walkforward import plot_walkforward_metrics

plot_walkforward_metrics(metrics_xgb, metrics_lstm, summary, Path("notebooks/figures/wf_metrics.png"))
print("Saved notebooks/figures/wf_metrics.png")

import wandb

run = wandb.init(
  project="mnq-microstructure",
  name="wf-summary",
  group="walkforward-summary",
  job_type="aggregate",
  settings=wandb.Settings(init_timeout=300),
)
wandb.log({"walkforward_summary": wandb.Table(dataframe=summary)})
wandb.log({"wf_metrics_png": wandb.Image("notebooks/figures/wf_metrics.png")})
run.finish()

## Trading simulation (Day 26)

Naive simulator: `signed_score = p(+1) - p(-1)` → position with threshold 0.2, half-spread costs on every change. Run `scripts/run_simulation.py` to refresh `data/results/*_sim.parquet`.

In [ ]:
from pathlib import Path

sim_xgb = pd.read_parquet("data/results/xgb_sim.parquet")
sim_lstm = pd.read_parquet("data/results/lstm_sim.parquet")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sim_xgb["ts_event"], sim_xgb["cum_pnl"], label="XGBoost", linewidth=0.8, alpha=0.9)
ax.plot(sim_lstm["ts_event"], sim_lstm["cum_pnl"], label="LSTM", linewidth=0.8, alpha=0.9)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Equity curves — naive sim (threshold=0.2, half-spread costs)")
ax.set_xlabel("ts_event")
ax.set_ylabel("cumulative PnL ($)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(Path("notebooks/figures/fig04_equity_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## Latency + threshold sweep (Day 27)

Fill delay: decision at bar `t` fills at `t + delay` (paying that bar's quotes). Commission $0.35/contract/side. Run `scripts/run_simulation.py` to refresh `data/results/sim_sweep.parquet`.

In [ ]:
sweep = pd.read_parquet("data/results/sim_sweep.parquet")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, model in zip(axes, ["xgb", "lstm"]):
    sub = sweep[sweep["model"] == model]
    pivot = sub.pivot(
        index="fill_delay_bars", columns="entry_threshold", values="sharpe"
    )
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=-15, vmax=5)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{x:.1f}" for x in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("entry_threshold")
    ax.set_ylabel("fill_delay_bars")
    ax.set_title(f"{model.upper()} — Sharpe")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="Sharpe")
fig.suptitle("Sharpe vs fill latency and entry threshold", y=1.02)
fig.tight_layout()
fig.savefig(Path("notebooks/figures/fig05_latency_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

# XGB churns on low thresholds → Sharpe craters as delay rises (stale fills on noisy signals).
# LSTM trades less; heatmap is flatter but still negative — weak edge + costs dominate.

## Latency benchmark (Day 28)

Per-stage p50/p95/p99 in microseconds. Run `scripts/benchmark_latency.py` → `data/results/latency.csv`.

In [ ]:
lat = pd.read_csv("data/results/latency.csv")
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(lat))
w = 0.25
ax.bar(x - w, lat["p50_us"], width=w, label="p50")
ax.bar(x, lat["p95_us"], width=w, label="p95")
ax.bar(x + w, lat["p99_us"], width=w, label="p99")
ax.set_xticks(x)
ax.set_xticklabels(lat["stage"], rotation=15, ha="right")
ax.set_ylabel("latency (µs)")
ax.set_title("Per-stage latency")
ax.legend()
ax.axhline(100_000, color="gray", linestyle="--", linewidth=0.8, label="100ms bar budget")
fig.tight_layout()
fig.savefig(Path("notebooks/figures/fig06_inference_p99.png"), dpi=150, bbox_inches="tight")
plt.show()

# Typical findings (re-run benchmark_latency.py on your machine to refresh):
# - XGB inference p99 ~2.5 ms vs LSTM ~5 ms → LSTM ~2× slower (not 10–100× on CPU).
# - Feature compute p99 ~4 ms (~4% of a 100 ms bar) — batch Polars pipeline on rolling
#   windows; incremental updates in prod would be faster. Still the largest stage at p99.
# - All stages fit inside a 100 ms bar budget on this hardware; inference is not the blocker.